In [1]:
"""
SimCLR 质谱二分类微调入口（ main.ipynb ）
- 核心逻辑已剥离至 lib/ 目录下以提高可维护性
- 本文件只保留运行入口、配置项与流程编排
"""

import os

import sys
import torch
from pathlib import Path
from datetime import datetime

# ==================== 1. 环境与路径配置 ====================
base_dir = Path(__file__).parent if '__file__' in locals() else Path.cwd()
project_root = base_dir.parent if base_dir.name == 'simclr_finetune' else base_dir

# 确保项目根目录在 python 模块搜索路径中
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 导入 simclr_finetune 模块
from simclr_finetune.lib import (
    load_pretrained_encoder,
    save_model_checkpoint,
    BinaryClassifier,
    prepare_finetune_dataset,
    train_binary_classifier,
    evaluate_and_record_predictions,
    evaluate_positive_per_smiles,
    export_results_to_excel,
    plot_comprehensive_results,
    plot_confusion_matrices
)

# 超参数配置
batch_size = 128
lr = 0.001
epochs = 100
patience = 15
freeze_encoder = True
pos_weight = 1.55
threshold = 0.5  # 推理判定阈值设置: 0.5

# 设置推理与微调设备 (优先 CUDA)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("=" * 60, flush=True)
print("SimCLR 质谱二分类微调管线", flush=True)
print("=" * 60, flush=True)
print(f"运行设备: {device.upper()}", flush=True)
if device == 'cuda':
    print(f"显卡型号: {torch.cuda.get_device_name(0)}")
print(f"推理判定阈值: {threshold}", flush=True)

# 数据与模型文件路径
pos_msp_path = base_dir / "data_source" / "阳性-含CanonicalSMILES-5类骨架(4).msp"
neg_msp_path = base_dir / "data_source" / "阴性(4).msp"
encoder_path = project_root / "simclr_pretrain" / "pretrained_encoder_final.pt"

print(f"正样本路径: {pos_msp_path}", flush=True)
print(f"负样本路径: {neg_msp_path}", flush=True)
print(f"编码器路径: {encoder_path}", flush=True)

# ==================== 2. 数据准备与划分 ====================
print("\n" + "=" * 60, flush=True)
print("Step 1: 数据加载、预处理与划分", flush=True)
print("=" * 60, flush=True)
train_loader, train_eval_loader, val_loader, test_loader, meta = prepare_finetune_dataset(
    pos_msp_path=pos_msp_path,
    neg_msp_path=neg_msp_path,
    batch_size=batch_size,
    test_size=0.15,
    val_size=0.15,
    random_state=42,
    use_cuda=(device == 'cuda')
)

print(f"  训练集批次数: {len(train_loader)}", flush=True)
print(f"  验证集批次数: {len(val_loader)}", flush=True)
print(f"  测试集批次数: {len(test_loader)}", flush=True)

# ==================== 3. 模型构建与权重加载 ====================
print("\n" + "=" * 60, flush=True)
print("Step 2: 构建模型并加载预训练编码器", flush=True)
print("=" * 60, flush=True)
encoder = load_pretrained_encoder(
    encoder_path=encoder_path,
    input_dim=561,
    hidden_dim=256,
    device=device
)
model = BinaryClassifier(encoder, input_dim=256, freeze_encoder=freeze_encoder)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  可训练参数量: {trainable_params:,}", flush=True)

# ==================== 4. 微调训练 ====================
print("\n" + "=" * 60, flush=True)
print("Step 3: 开始二分类微调训练", flush=True)
print("=" * 60, flush=True)
start_time = datetime.now()
model, history = train_binary_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=epochs,
    lr=lr,
    patience=patience,
    pos_weight=pos_weight
)
elapsed = datetime.now() - start_time
print(f"  [OK] 训练完成，总用时: {str(elapsed).split('.')[0]}", flush=True)

# ==================== 5. 综合评估 ====================
print("\n" + "=" * 60, flush=True)
print(f"Step 4: 模型性能评估 (判定阈值={threshold})", flush=True)
print("=" * 60, flush=True)
train_res = evaluate_and_record_predictions(model, train_eval_loader, sample_names=meta['train_sample_names'], device=device, threshold=threshold)
print("  [DEBUG] Calling for val_loader", flush=True)
val_res = evaluate_and_record_predictions(model, val_loader, sample_names=meta['val_sample_names'], device=device, threshold=threshold)
print("  [DEBUG] Calling for test_loader", flush=True)
test_res = evaluate_and_record_predictions(model, test_loader, sample_names=meta['test_sample_names'], device=device, threshold=threshold)

print(f"  训练集 | Accuracy: {train_res['accuracy']:.2%} | AUC: {train_res['auc']:.4f}", flush=True)
print(f"  验证集 | Accuracy: {val_res['accuracy']:.2%} | AUC: {val_res['auc']:.4f}", flush=True)
print(f"  测试集 | Accuracy: {test_res['accuracy']:.2%} | AUC: {test_res['auc']:.4f}", flush=True)

smiles_res = evaluate_positive_per_smiles(
    test_indices=meta['test_idx'],
    smiles_all=meta['smiles_all'],
    labels_all=meta['y'],
    probs=test_res['probs'],
    preds=test_res['preds'],
    threshold=threshold
)



SimCLR 质谱二分类微调管线
运行设备: CUDA
显卡型号: NVIDIA GeForce RTX 3070 Laptop GPU
推理判定阈值: 0.5
正样本路径: D:\UserFiles\Documents\PyCharm\cannabinoids\simclr_finetune\data_source\阳性-含CanonicalSMILES-5类骨架(4).msp
负样本路径: D:\UserFiles\Documents\PyCharm\cannabinoids\simclr_finetune\data_source\阴性(4).msp
编码器路径: D:\UserFiles\Documents\PyCharm\cannabinoids\simclr_pretrain\pretrained_encoder_final.pt

Step 1: 数据加载、预处理与划分
  正在解析 MSP 文件: 阳性-含CanonicalSMILES-5类骨架(4).msp...
  [OK] 成功解析出 1355 个化合物谱图
  正在解析 MSP 文件: 阴性(4).msp...
  [OK] 成功解析出 2097 个化合物谱图
  训练集批次数: 19
  验证集批次数: 5
  测试集批次数: 5

Step 2: 构建模型并加载预训练编码器
  [OK] 成功加载预训练权重: D:\UserFiles\Documents\PyCharm\cannabinoids\simclr_pretrain\pretrained_encoder_final.pt
  可训练参数量: 41,601

Step 3: 开始二分类微调训练
训练执行设备: cuda
  [INFO] 启用正样本损失加权 pos_weight = 1.55
Epoch   1 | Train Loss: 0.3819 | Train Acc: 90.61% | Val Loss: 0.2512 | Val Acc: 95.21% | LR: 1.00e-03
Epoch   6 | Train Loss: 0.1318 | Train Acc: 97.07% | Val Loss: 0.1999 | Val Acc: 94.64% | LR: 1.00e-03
Epoch  11 | Train

In [2]:
# ==================== 6. 结果保存与导出 ====================
print("\n" + "=" * 60, flush=True)
print("Step 5: 保存模型与导出评估结果", flush=True)
print("=" * 60, flush=True)
# 为本次运行创建统一的时间戳专属输出目录
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = base_dir / f"results_{timestamp}"
run_dir.mkdir(parents=True, exist_ok=True)

# 1. 保存模型权重至专属结果目录
print("  [DEBUG] Calling save_model_checkpoint...", flush=True)
saved_path = save_model_checkpoint({
    'encoder_state_dict': encoder.state_dict(),
    'classifier_state_dict': model.classifier.state_dict(),
    'history': history,
    'test_metrics': test_res,
}, run_dir)

# 2. 导出 CSV / Excel 评估明细表格至专属结果目录
print("  [DEBUG] Calling export_results_to_excel...", flush=True)
export_results_to_excel(
    history=history,
    model=model,
    train_results=train_res,
    val_results=val_res,
    test_results=test_res,
    smiles_results=smiles_res,
    meta=meta,
    output_dir=run_dir
)

# 3. 保存所有可视化图表至专属结果目录
print("  [DEBUG] Calling plot_comprehensive_results...", flush=True)
plot_comprehensive_results(history, train_res, val_res, test_res, smiles_res, run_dir)
print("  [DEBUG] Calling plot_confusion_matrices...", flush=True)
plot_confusion_matrices(train_res, val_res, test_res, run_dir)

print(f"\n  [OK] 所有输出结果（模型权重 .pt、评估表格 CSV 及图表 PNG）已统一保存至:", flush=True)
print(f"       --> {run_dir}", flush=True)
print("\n" + "=" * 60, flush=True)
print("微调管线全部顺利完成！", flush=True)
print("=" * 60, flush=True)



Step 5: 保存模型与导出评估结果
  [DEBUG] Calling save_model_checkpoint...
  [OK] 模型权重已保存至全新时间戳文件: D:\UserFiles\Documents\PyCharm\cannabinoids\simclr_finetune\results_20260723_195942\binary_classifier_20260723_195942.pt
  [DEBUG] Calling export_results_to_excel...
  [OK] 评估表格 CSV 已全部导出至目录: D:\UserFiles\Documents\PyCharm\cannabinoids\simclr_finetune\results_20260723_195942
  [DEBUG] Calling plot_comprehensive_results...
  [OK] 评估图表已保存至: D:\UserFiles\Documents\PyCharm\cannabinoids\simclr_finetune\results_20260723_195942
  [DEBUG] Calling plot_confusion_matrices...
  [OK] 混淆矩阵热力图已保存: D:\UserFiles\Documents\PyCharm\cannabinoids\simclr_finetune\results_20260723_195942\confusion_matrix.png

  [OK] 所有输出结果（模型权重 .pt、评估表格 CSV 及图表 PNG）已统一保存至:
       --> D:\UserFiles\Documents\PyCharm\cannabinoids\simclr_finetune\results_20260723_195942

微调管线全部顺利完成！
